# Phase 2 -- QLoRA + DoRA fine-tuning: Qwen3-8B on NHTSA complaint extraction

Runs on **Colab or Kaggle free-tier GPU only** (T4/P100, 16GB VRAM) -- per the project's hardware contract (see `docs/blueprint.md`), this never runs on the local machine (8GB RAM, RTX 3050 4GB VRAM).

## Locked hyperparameters (options-first proposal, confirmed before this notebook was written)

| Hyperparameter | Value | Why |
|---|---|---|
| LoRA rank (r) | 16 | Middle ground for a 4-field structured-extraction task -- enough capacity without overfitting a small dataset |
| LoRA alpha | 32 | Standard alpha = 2×r scaling convention |
| DoRA | `use_dora=True` | Same-cost quality upgrade over plain LoRA, per blueprint.md |
| Learning rate | 2e-4 | Standard QLoRA default for 7-8B models |
| Epochs | 3 | Small-dataset QLoRA norm (2-4 epochs); eval loss logged per epoch to watch for overfitting. Confirmed twice now (v1 and v2 runs) that epoch 3 reliably regresses -- epoch 2 is the checkpoint to keep |
| Batch size | 4 per step, **4 gradient accumulation steps -> effective batch 16** | Batch 4 alone is small/noisy; accumulating to an effective 16 gives more stable gradients. Confirmed to fit a 16GB T4 across two real runs |
| Max sequence length | **768** (raised from an initial 512 guess) | Checked against the *real* Qwen3 tokenizer over the training set formatted as instruction+narrative+JSON -- 768 covers 99.9% of examples (`scripts/check_seq_lengths.py`) |

## Data

Needs `data/processed/train.jsonl` and `data/processed/eval.jsonl` from Phase 1 (see `docs/label-strategy.md` for how those labels were derived, including the two post-Phase-3 targeted fixes to the training set's severity-tier balance). The data-loading cell below auto-detects an attached Kaggle Dataset containing both files -- **any dataset name works**, no path editing needed. On Colab, run that cell by itself to get a working upload button.

## What this notebook does NOT do

It does not evaluate baseline-vs-fine-tuned accuracy (that's Phase 3) and does not quantize to GGUF (Phase 4).

## 1. Install dependencies (Colab/Kaggle only -- never run locally)

In [ ]:
%%capture
!pip install unsloth
# unsloth pulls in the pinned-compatible transformers/peft/trl/bitsandbytes itself

## 2. Load Qwen3-8B in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 768  # see length-check note above -- covers 99.9% of the training set untruncated

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",  # Unsloth's pre-quantized 4-bit Qwen3-8B (instruct, not -Base)
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    dtype = None,  # auto-detect (bf16 on T4/P100-class GPUs that support it, else fp16)
)

## 3. Attach LoRA + DoRA adapters

In [ ]:
LORA_R = 16
LORA_ALPHA = 32

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,  # 0 is Unsloth's recommended/optimized setting
    bias = "none",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_dora = True,  # blueprint.md: QLoRA + DoRA, same VRAM cost as plain LoRA
    use_gradient_checkpointing = "unsloth",  # Unsloth's memory-efficient checkpointing, needed to fit a 16GB T4
    random_state = 42,
)
model.print_trainable_parameters()

## 4. Load Phase 1's train/eval data

In [ ]:
import glob, json, os, shutil

def find_kaggle_dataset(*required_files):
    """Search /kaggle/input/*/ for a folder containing all of required_files --
    works no matter what the attached Dataset is named, so re-uploading under a new
    name never requires editing this notebook."""
    for candidate in sorted(glob.glob("/kaggle/input/*/")):
        if all(os.path.exists(os.path.join(candidate, f)) for f in required_files):
            return candidate.rstrip("/")
    return None

if not (os.path.exists("train.jsonl") and os.path.exists("eval.jsonl")):
    kaggle_dir = find_kaggle_dataset("train.jsonl", "eval.jsonl")
    if kaggle_dir:
        shutil.copy(os.path.join(kaggle_dir, "train.jsonl"), "train.jsonl")
        shutil.copy(os.path.join(kaggle_dir, "eval.jsonl"), "eval.jsonl")
        print(f"found and copied train.jsonl/eval.jsonl from {kaggle_dir}")
    else:
        try:
            from google.colab import files
            print("Upload data/processed/train.jsonl and data/processed/eval.jsonl from the repo:")
            files.upload()
        except ImportError:
            raise RuntimeError(
                "train.jsonl/eval.jsonl not found locally, and no /kaggle/input/*/ folder "
                "contains both files. On Kaggle: attach a Dataset with both files in it -- "
                "any dataset name works. On Colab: rerun this cell (by itself, not via "
                "'Run All') to get a working upload button."
            )

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_rows = load_jsonl("train.jsonl")
eval_rows = load_jsonl("eval.jsonl")
print(f"train: {len(train_rows)}  eval: {len(eval_rows)}")

## 5. Format as chat-template training text

Thinking mode disabled (`enable_thinking=False`) -- per blueprint.md, this task wants fast deterministic extraction, not exploratory chain-of-thought bleeding into the JSON output.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an automotive safety complaint analyst. Given a raw consumer complaint "
    "about a vehicle, extract a structured JSON object with exactly these fields: "
    'component (string), defect_type (string), safety_risk ("yes" or "no"), '
    'severity ("low", "medium", or "high"). Respond with only the JSON object.'
)

def to_chat_text(row):
    target = json.dumps({
        "component": row["component"],
        "defect_type": row["defect_type"],
        "safety_risk": row["safety_risk"],
        "severity": row["severity"],
    })
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Complaint:\n{row['narrative']}"},
        {"role": "assistant", "content": target},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

train_dataset = Dataset.from_list([{"text": to_chat_text(r)} for r in train_rows])
eval_dataset = Dataset.from_list([{"text": to_chat_text(r)} for r in eval_rows])
print(train_dataset[0]["text"][:500])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir = "outputs",
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    gradient_accumulation_steps = 4,   # effective batch size 16 -- confirmed with user before locking
    num_train_epochs = 3,
    learning_rate = 2e-4,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.03,
    optim = "adamw_8bit",              # memory-efficient optimizer, standard for QLoRA
    logging_steps = 10,
    eval_strategy = "epoch",           # eval loss logged after every epoch -- overfitting check, see cell below
    save_strategy = "epoch",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_text_field = "text",
    packing = False,
    seed = 42,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = training_args,
)

trainer_stats = trainer.train()

## 7. Overfitting check -- train vs. eval loss per epoch

With only 800 training examples over 3 epochs, watch for eval loss climbing while train loss keeps dropping -- that's the overfitting signal, flagged as a real risk to check for (not assume away) before calling this run done.

In [ ]:
history = trainer.state.log_history
train_losses = [(h["epoch"], h["loss"]) for h in history if "loss" in h]
eval_losses = [(h["epoch"], h["eval_loss"]) for h in history if "eval_loss" in h]

print("epoch  train_loss (last logged that epoch)")
last_by_epoch = {}
for ep, loss in train_losses:
    last_by_epoch[round(ep)] = loss
for ep, loss in sorted(last_by_epoch.items()):
    print(f"{ep:>5}  {loss:.4f}")

print("\nepoch  eval_loss")
for ep, loss in eval_losses:
    print(f"{round(ep):>5}  {loss:.4f}")

if len(eval_losses) >= 2 and eval_losses[-1][1] > eval_losses[-2][1]:
    print("\nFLAG: eval loss increased on the last epoch -- possible overfitting. "
          "Consider using the earlier checkpoint (outputs/checkpoint-*) instead of the final one.")
else:
    print("\nEval loss did not increase between the last two epochs -- no overfitting flag.")

## 8. Save the adapter

In [ ]:
from datetime import datetime

# Timestamped, not a fixed name -- every run gets its own folder automatically, so
# re-running this notebook never requires remembering to rename anything by hand
# (and never risks silently overwriting a previous run's adapter).
ADAPTER_DIR = f"qwen3-8b-automotive-complaint-lora-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import shutil
shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
print(f"Saved and zipped: {ADAPTER_DIR}.zip -- download this and keep it for Phase 3/4.")

try:
    from google.colab import files
    files.download(f"{ADAPTER_DIR}.zip")
except ImportError:
    print("Not on Colab -- grab the zip from the Kaggle notebook's output files panel instead.")

## Next: Phase 3

Evaluate baseline (zero-shot Qwen3-8B) vs. this fine-tuned adapter on `data/processed/eval.jsonl`, measuring JSON validity rate and field-level accuracy -- with `safety_risk` precision/recall and per-tier `severity` accuracy reported separately, not just overall accuracy (per blueprint.md Section 5a).